In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import pickle
from datetime import datetime

# ML imports
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

# XGBoost
import xgboost as xgb

warnings.filterwarnings('ignore')
print("Imports complete")

Imports complete


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
football_api_key = user_secrets.get_secret("MY_API_KEY")

In [3]:
# =============================================================================
# CELL 2: CONFIGURABLE PARAMETERS
# =============================================================================
# Adjust these parameters to experiment with different configurations

# ----- Data Parameters -----
ROLLING_WINDOW = 5          # Number of previous games for rolling stats (try: 3, 5, 10)
MIN_MATCHES_REQUIRED = 3    # Minimum matches before making predictions

# ----- Train/Test Split -----
TEST_SEASON = '2025-2026'     # Season to use as test set

# ----- XGBoost Parameters -----
XGBOOST_PARAMS = {
    'objective': 'multi:softprob',      # QUAN TRỌNG: Để lấy xác suất chạy Monte Carlo
    'num_class': 3,                    # Phải có cái này: 0 (Away), 1 (Draw), 2 (Home)
    'eval_metric': ['mlogloss', 'merror'],         # Thước đo chuẩn cho phân loại đa lớp
    'max_depth': 4,                    
    'learning_rate': 0.05,             
    'n_estimators': 500,               
    'min_child_weight': 10,            
    'subsample': 0.8,                  
    'colsample_bytree': 0.8,           
    'reg_alpha': 0.1,                  
    'reg_lambda': 1.0,                 
    'random_state': 42,
    'n_jobs': -1
}
# ----- Early Stopping -----
EARLY_STOPPING_ROUNDS = 50  # Stop if no improvement for N rounds
VALIDATION_SPLIT = 0.2     # Fraction of training data for validation

# ----- Over/Under Thresholds -----
OU_THRESHOLDS = [8.5, 9.5, 10.5, 11.5, 12.5]

print("Parameters configured:")
print(f"  Rolling window: {ROLLING_WINDOW}")
print(f"  Test season: {TEST_SEASON}")
print(f"  XGBoost max_depth: {XGBOOST_PARAMS['max_depth']}")
print(f"  XGBoost learning_rate: {XGBOOST_PARAMS['learning_rate']}")

Parameters configured:
  Rolling window: 5
  Test season: 2025-2026
  XGBoost max_depth: 4
  XGBoost learning_rate: 0.05


In [4]:
import requests
import pandas as pd

def get_upcoming_fixtures(api_key):
    # Endpoint lấy các trận đấu của Premier League (Mã giải đấu là 'PL' hoặc ID 2021)
    url = "https://api.football-data.org/v4/competitions/PL/matches?status=SCHEDULED"
    headers = { 'X-Auth-Token': api_key }
    
    try:
        response = requests.get(url, headers=headers)
        data = response.json()
        
        fixtures = []
        for match in data['matches']:
            fixtures.append({
                'Date': match['utcDate'],
                'HomeTeam': match['homeTeam']['shortName'], # Hoặc .name tùy theo mapping
                'AwayTeam': match['awayTeam']['shortName'],
                'Season': '2025-26',
                'FTR': None # Trận chưa đá nên kết quả để trống
            })
            
        df_fixtures = pd.DataFrame(fixtures)
        # Chuyển đổi Date sang datetime và bỏ phần giờ để khớp với CSV
        df_fixtures['Date'] = pd.to_datetime(df_fixtures['Date']).dt.tz_localize(None)
        
        return df_fixtures
    
    except Exception as e:
        print(f"❌ Lỗi khi lấy API: {e}")
        return pd.DataFrame()

In [5]:
# =============================================================================
# CELL 3: Load Data from Football-Data.co.uk
# =============================================================================
SEASONS = {
    # '2000-01': '0001',
    # '2001-02': '0102',
    # '2002-03': '0203',
    # '2003-04': '0304',
    # '2004-05': '0405',
    '2005-06': '0506',
    '2006-07': '0607',
    '2007-08': '0708',
    '2008-09': '0809',
    '2009-10': '0910',
    '2010-11': '1011',
    '2011-12': '1112',
    '2012-13': '1213',
    '2013-14': '1314',
    '2014-15': '1415',
    '2015-16': '1516',
    '2016-17': '1617',
    '2017-18': '1718',
    '2018-19': '1819',
    '2019-20': '1920',
    '2020-21': '2021',
    '2021-22': '2122',
    '2022-23': '2223',
    '2023-24': '2324',
    '2024-25': '2425',
    '2025-26': '2526'
}

BASE_URL = 'https://www.football-data.co.uk/mmz4281/{code}/E0.csv'

COLS = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
        'HS', 'AS', 'HY', 'AY', 'HST', 'AST', 'HC', 'AC', 'HR', 'AR', 'HF', 'AF' ]

print("Loading data from Football-Data.co.uk...\n")
dfs = []
for season_name, season_code in SEASONS.items():
    url = BASE_URL.format(code=season_code)
    try:
        df = pd.read_csv(url, encoding='utf-8')
        available_cols = [c for c in COLS if c in df.columns]
        df = df[available_cols].copy()
        df['Season'] = season_name
        print(f"  {season_name}: {len(df)} matches")
        dfs.append(df)
    except Exception as e:
        print(f"  {season_name}: Failed - {e}")

df = pd.concat(dfs, ignore_index=True)
df = df.dropna(subset=['Date', 'HomeTeam'])
try:
    # Gọi hàm lấy Fixtures mà chúng ta vừa viết (đảm bảo hàm này đã được định nghĩa trước)
    df_upcoming = get_upcoming_fixtures(football_api_key) 
    
    if not df_upcoming.empty:
        # df_upcoming = df_upcoming.sort_values('Date')
        df_upcoming = df_upcoming.head(10)
        
        team_mapping = {
            'Nottingham': "Nott'm Forest", 'Brighton Hove': 'Brighton',
            'Leeds United': 'Leeds', 'Wolverhampton': 'Wolves'
            # Kiểm tra lại tên chuẩn trong CSV của bạn
        }
        df_upcoming['HomeTeam'] = df_upcoming['HomeTeam'].replace(team_mapping)
        df_upcoming['AwayTeam'] = df_upcoming['AwayTeam'].replace(team_mapping)
        
        # Nối vào df chính
        df = pd.concat([df, df_upcoming], ignore_index=True)
        print(f"\n✅ Added {len(df_upcoming)} upcoming fixtures from API.")
except Exception as e:
    print(f"\n⚠️ Could not add API fixtures: {e}")

df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.sort_values('Date').reset_index(drop=True)

print(f"\nTotal matches: {len(df)}")
print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")

Loading data from Football-Data.co.uk...

  2005-06: 380 matches
  2006-07: 380 matches
  2007-08: 380 matches
  2008-09: 380 matches
  2009-10: 380 matches
  2010-11: 380 matches
  2011-12: 380 matches
  2012-13: 380 matches
  2013-14: 380 matches
  2014-15: 381 matches
  2015-16: 380 matches
  2016-17: 380 matches
  2017-18: 380 matches
  2018-19: 380 matches
  2019-20: 380 matches
  2020-21: 380 matches
  2021-22: 380 matches
  2022-23: 380 matches
  2023-24: 380 matches
  2024-25: 380 matches
  2025-26: 332 matches

✅ Added 10 upcoming fixtures from API.

Total matches: 7942
Date range: 2005-08-13 to 2026-05-04


In [6]:
df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,HST,AST,HC,AC,HR,AR,HF,AF,Season
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,2.0,6.0,7.0,8.0,0.0,0.0,14.0,16.0,2005-06
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,5.0,5.0,8.0,6.0,0.0,0.0,15.0,14.0,2005-06
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,7.0,4.0,6.0,6.0,0.0,0.0,12.0,13.0,2005-06
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,8.0,3.0,3.0,6.0,0.0,0.0,13.0,11.0,2005-06
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,2.0,7.0,5.0,0.0,1.0,0.0,17.0,11.0,2005-06


In [7]:
df.tail(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,HST,AST,HC,AC,HR,AR,HF,AF,Season
7932,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7933,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7934,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7935,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7936,2026-05-02 14:00:00,Wolves,Sunderland,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7937,2026-05-02 16:30:00,Arsenal,Fulham,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7938,2026-05-03 13:00:00,Bournemouth,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7939,2026-05-03 14:30:00,Man United,Liverpool,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7940,2026-05-03 18:00:00,Aston Villa,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26
7941,2026-05-04 14:00:00,Chelsea,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-26


# ROLLING FEATURE

In [8]:
def create_rolling_features(df, n_games=5):
    # Đảm bảo sắp xếp theo thời gian
    data = df.sort_values('Date').copy()
    
    # Danh sách các chỉ số thô cần tính rolling
    metrics = {
        'goals': ['FTHG', 'FTAG'],
        'shots': ['HS', 'AS'],
        'shots_ot': ['HST', 'AST'],
        'corners': ['HC', 'AC'],
        'fouls': ['HF', 'AF'],
        'yellows': ['HY', 'AY'],
        'reds': ['HR', 'AR']
    }

    # Khởi tạo các cột kết quả với NaN
    for side in ['Home', 'Away']:
        for m in metrics.keys():
            data[f'{side}_Avg_{m}_{n_games}'] = np.nan
            data[f'{side}_Avg_{m}_Conceded_{n_games}'] = np.nan

    all_teams = pd.concat([data['HomeTeam'], data['AwayTeam']]).unique()

    for team in all_teams:
        # Lọc các trận của đội này
        team_mask = (data['HomeTeam'] == team) | (data['AwayTeam'] == team)
        team_df = data[team_mask].copy()
        
        # 1. Tạo một DataFrame TẠM THỜI chỉ chứa các con số để tính Rolling
        # Không đưa cột 'Date' vào đây
        rows = []
        for idx, row in team_df.iterrows():
            is_home = row['HomeTeam'] == team
            stats = {
                'goals': row['FTHG'] if is_home else row['FTAG'],
                'goals_conceded': row['FTAG'] if is_home else row['FTHG'],
                'shots': row['HS'] if is_home else row['AS'],
                'shots_conceded': row['AS'] if is_home else row['HS'], # Thêm cái này
                'shots_ot': row['HST'] if is_home else row['AST'],
                'shots_ot_conceded': row['AST'] if is_home else row['HST'], # VÀ CÁI NÀY
                'corners': row['HC'] if is_home else row['AC'],
                'fouls': row['HF'] if is_home else row['AF'],
                'yellows': row['HY'] if is_home else row['AY'],
                'reds': row['HR'] if is_home else row['AR']
            }
            rows.append(stats)
        
        # Tạo df chỉ gồm số, reset index để tránh lỗi Date ở Index
        temp_stats_df = pd.DataFrame(rows, index=team_df.index)
        
        # 2. Tính Rolling trên dữ liệu THUẦN SỐ
        roll_df = temp_stats_df.shift(1).rolling(window=n_games, min_periods=1).mean()
        
        # 3. Gán ngược lại vào DataFrame chính dựa trên Index
       # 3. Gán ngược lại vào DataFrame chính dựa trên Index
        for idx in team_df.index:
            is_home = data.at[idx, 'HomeTeam'] == team
            prefix = 'Home' if is_home else 'Away'
            
            # Gán toàn bộ các chỉ số đã tính trong roll_df
            for col in temp_stats_df.columns:
                # col ở đây bao gồm: goals, goals_conceded, shots, shots_ot, corners, ...
                if 'conceded' in col:
                    # Gán các chỉ số phòng ngự: shots_ot_conceded, goals_conceded...
                    data.at[idx, f'{prefix}_Avg_{col}_{n_games}'] = roll_df.at[idx, col]
                else:
                    # Gán các chỉ số tấn công: shots, shots_ot, corners...
                    data.at[idx, f'{prefix}_Avg_{col}_{n_games}'] = roll_df.at[idx, col]
            
            # QUAN TRỌNG: Để tính Defense_Ratio, chúng ta cần shots_ot bị đối phương sút
            # Trong stats ở bước 1, bạn chưa tính 'shots_ot_conceded'. Hãy thêm nó vào bước 1!
    return data

full_df = create_rolling_features(df)

In [9]:
full_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Away_Avg_yellows_5,Away_Avg_yellows_Conceded_5,Away_Avg_reds_5,Away_Avg_reds_Conceded_5,Home_Avg_goals_conceded_5,Home_Avg_shots_conceded_5,Home_Avg_shots_ot_conceded_5,Away_Avg_goals_conceded_5,Away_Avg_shots_conceded_5,Away_Avg_shots_ot_conceded_5
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
full_df.tail()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Away_Avg_yellows_5,Away_Avg_yellows_Conceded_5,Away_Avg_reds_5,Away_Avg_reds_Conceded_5,Home_Avg_goals_conceded_5,Home_Avg_shots_conceded_5,Home_Avg_shots_ot_conceded_5,Away_Avg_goals_conceded_5,Away_Avg_shots_conceded_5,Away_Avg_shots_ot_conceded_5
7937,2026-05-02 16:30:00,Arsenal,Fulham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1.6,NaN,0.0,NaN,1.00,10.40,3.4,0.8,12.0,3.8
7938,2026-05-03 13:00:00,Bournemouth,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1.6,NaN,0.2,NaN,1.20,13.20,3.2,0.8,11.6,5.0
7939,2026-05-03 14:30:00,Man United,Liverpool,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1.2,NaN,0.0,NaN,1.25,15.25,4.0,1.2,12.4,5.0
7940,2026-05-03 18:00:00,Aston Villa,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,2.0,NaN,0.0,NaN,2.20,13.00,5.2,2.0,11.4,4.0
7941,2026-05-04 14:00:00,Chelsea,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,...,2.0,NaN,0.0,NaN,2.20,10.80,6.4,0.8,11.2,3.6


# Month sin / cos

In [11]:
def smart_date_parser(date_str):
    if pd.isna(date_str):
        return pd.NaT
    # Các định dạng phổ biến trong dữ liệu Football-data
    for fmt in ('%d/%m/%y', '%d/%m/%Y', '%Y-%m-%d'):
        try:
            return pd.to_datetime(date_str, format=fmt)
        except (ValueError, TypeError):
            continue
    # Nếu không khớp cái nào, để Pandas tự đoán lần cuối
    return pd.to_datetime(date_str, errors='coerce')

# Áp dụng hàm cho cột Date
full_df['Date'] = full_df['Date'].apply(smart_date_parser)

# Sau đó nhớ sort lại để đảm bảo thứ tự các trận đấu
full_df = full_df.sort_values('Date').reset_index(drop=True)
full_df['Month'] = full_df['Date'].dt.month

In [12]:
import numpy as np

# --- BƯỚC 1: Xử lý Month sang Sin/Cos ---
# Giúp mô hình hiểu tính chu kỳ (Tháng 12 gần Tháng 1)
full_df['Month_Sin'] = np.sin(2 * np.pi * full_df['Month'] / 12)
full_df['Month_Cos'] = np.cos(2 * np.pi * full_df['Month'] / 12)

# --- BƯỚC 2: Tạo cột Season (Cần thiết để reset Matchweek mỗi năm) ---
# Mùa giải Ngoại hạng Anh thường bắt đầu từ tháng 8
full_df['Season'] = full_df['Date'].apply(lambda x: f"{x.year}-{x.year+1}" if x.month >= 8 else f"{x.year-1}-{x.year}")

# --- BƯỚC 3: Tính Matchweek tự động ---
def calculate_matchweek(df):
    df = df.sort_values(['Season', 'Date']).copy()
    
    # Dictionary lưu số trận đã đá của từng đội trong từng mùa
    counts = {} 
    mws = []
    
    for idx, row in df.iterrows():
        s = row['Season']
        h = row['HomeTeam']
        a = row['AwayTeam']
        
        if s not in counts:
            counts[s] = {}
            
        # Lấy số trận hiện tại của 2 đội, mặc định là 0
        h_played = counts[s].get(h, 0)
        a_played = counts[s].get(a, 0)
        
        # Matchweek là số trận lớn nhất của 2 đội + 1 (để xử lý cả trận đá bù)
        current_mw = max(h_played, a_played) + 1
        mws.append(current_mw)
        
        # Cập nhật số trận đã đá cho trận tiếp theo
        counts[s][h] = h_played + 1
        counts[s][a] = a_played + 1
        
    df['Matchweek'] = mws
    return df

full_df = calculate_matchweek(full_df)

# Ranking 

In [13]:
def add_league_rank_features(df):
    # Sắp xếp theo mùa giải và thời gian
    df = df.sort_values(['Season', 'Date']).copy()
    
    # Lưu trữ trạng thái bảng xếp hạng: {Season: {Team: {'pts': 0, 'gd': 0, 'gs': 0}}}
    season_stats = {}
    
    home_ranks = []
    away_ranks = []

    for idx, row in df.iterrows():
        s = row['Season']
        h = row['HomeTeam']
        a = row['AwayTeam']
        
        if s not in season_stats:
            season_stats[s] = {}
            
        # Khởi tạo đội bóng nếu chưa có trong mùa này
        for team in [h, a]:
            if team not in season_stats[s]:
                season_stats[s][team] = {'pts': 0, 'gd': 0, 'gs': 0}
        
        # --- BƯỚC 1: Tính Rank TRƯỚC trận đấu ---
        # Chuyển stats hiện tại thành list để sort
        current_table = []
        for team, stats in season_stats[s].items():
            current_table.append({
                'team': team,
                'pts': stats['pts'],
                'gd': stats['gd'],
                'gs': stats['gs']
            })
        
        # Sort theo: Điểm > Hiệu số > Số bàn thắng
        current_table.sort(key=lambda x: (x['pts'], x['gd'], x['gs']), reverse=True)
        
        # Tạo map Team -> Vị trí (Rank)
        rank_map = {item['team']: i + 1 for i, item in enumerate(current_table)}
        
        # Gán rank cho trận hiện tại (nếu là trận đầu tiên của mùa thì mặc định là giữa bảng)
        home_ranks.append(rank_map.get(h, 10))
        away_ranks.append(rank_map.get(a, 10))
        
        # --- BƯỚC 2: Cập nhật kết quả SAU trận đấu để tính cho các vòng sau ---
        if pd.notna(row['FTR']):
            fthg, ftag = row['FTHG'], row['FTAG']
            
            # Cập nhật điểm
            if row['FTR'] == 'H':
                season_stats[s][h]['pts'] += 3
            elif row['FTR'] == 'A':
                season_stats[s][a]['pts'] += 3
            else:
                season_stats[s][h]['pts'] += 1
                season_stats[s][a]['pts'] += 1
            
            # Cập nhật hiệu số (GD) và bàn thắng (GS)
            season_stats[s][h]['gd'] += (fthg - ftag)
            season_stats[s][a]['gd'] += (ftag - fthg)
            season_stats[s][h]['gs'] += fthg
            season_stats[s][a]['gs'] += ftag
            
    df['Home_Rank'] = home_ranks
    df['Away_Rank'] = away_ranks
    df['Rank_Diff'] = df['Home_Rank'] - df['Away_Rank'] # Độ chênh lệch thứ hạng
    
    return df

# Thực hiện tính toán
full_df = add_league_rank_features(full_df)

# Kiểm tra thử kết quả
print(full_df[['Date', 'HomeTeam', 'AwayTeam', 'Matchweek', 'Home_Rank', 'Away_Rank']].tail(10))

                    Date     HomeTeam        AwayTeam  Matchweek  Home_Rank  \
7932 2026-04-27 19:00:00   Man United       Brentford         34          3   
7933 2026-05-01 19:00:00        Leeds         Burnley         35         15   
7934 2026-05-02 14:00:00    Brentford        West Ham         35          9   
7935 2026-05-02 14:00:00    Newcastle        Brighton         35         14   
7936 2026-05-02 14:00:00       Wolves      Sunderland         34         20   
7937 2026-05-02 16:30:00      Arsenal          Fulham         34          2   
7938 2026-05-03 13:00:00  Bournemouth  Crystal Palace         35          7   
7939 2026-05-03 14:30:00   Man United       Liverpool         35          3   
7940 2026-05-03 18:00:00  Aston Villa       Tottenham         34          4   
7941 2026-05-04 14:00:00      Chelsea   Nott'm Forest         35          8   

      Away_Rank  
7932          9  
7933         19  
7934         17  
7935          6  
7936         11  
7937         12  
7938

# SHOT AND DEFENSE 

In [14]:
# Đảm bảo tên cột khớp với output của hàm Rolling (thường là viết thường nếu bạn dùng dict stats)
# Giả sử hàm Rolling của bạn trả về các cột như: Home_Avg_shots_5, Home_Avg_shots_ot_5...

# 1. Shot Efficiency (Hiệu quả dứt điểm - Tấn công)
full_df['Home_Shot_Efficiency'] = full_df['Home_Avg_shots_ot_5'] / (full_df['Home_Avg_shots_5'] + 1e-5)
full_df['Away_Shot_Efficiency'] = full_df['Away_Avg_shots_ot_5'] / (full_df['Away_Avg_shots_5'] + 1e-5)

# 2. Defense Factor (Hệ số phòng ngự)
# Lưu ý: "Home_Avg_goals_conceded_5" là số bàn thua
# Chúng ta cần chia cho số cú sút trúng đích mà ĐỐI THỦ gây ra (đã được tính trong hàm rolling)
full_df['Home_Defense_Ratio'] = full_df['Home_Avg_goals_conceded_5'] / (full_df['Home_Avg_shots_ot_conceded_5'] + 1e-5)
full_df['Away_Defense_Ratio'] = full_df['Away_Avg_goals_conceded_5'] / (full_df['Away_Avg_shots_ot_conceded_5'] + 1e-5)

# 3. Pressure Index (Chỉ số áp lực - Kết hợp Phạt góc)
full_df['Home_Attack_Pressure'] = (full_df['Home_Avg_shots_ot_5'] * 0.7) + (full_df['Home_Avg_corners_5'] * 0.3)
full_df['Away_Attack_Pressure'] = (full_df['Away_Avg_shots_ot_5'] * 0.7) + (full_df['Away_Avg_corners_5'] * 0.3)

# H2H HISTORY

In [15]:
# def add_h2h_features(df):
#     h2h_map = {} # Lưu trữ cặp đội: [Thắng_Đội_A, Thắng_Đội_B, Hòa]
    
#     h2h_home_wins = []
#     h2h_away_wins = []
#     h2h_draws = []
    
#     for idx, row in df.iterrows():
#         teams = tuple(sorted([row['HomeTeam'], row['AwayTeam']]))
#         team_h = row['HomeTeam']
#         team_a = row['AwayTeam']
        
#         if teams not in h2h_map:
#             h2h_map[teams] = {team_h: 0, team_a: 0, 'draw': 0}
        
#         # Lấy dữ liệu trước trận
#         current_h2h = h2h_map[teams]
#         h2h_home_wins.append(current_h2h.get(team_h, 0))
#         h2h_away_wins.append(current_h2h.get(team_a, 0))
#         h2h_draws.append(current_h2h['draw'])
        
#         # Cập nhật kết quả sau trận
#         if row['FTR'] == 'H':
#             h2h_map[teams][team_h] += 1
#         elif row['FTR'] == 'A':
#             h2h_map[teams][team_a] += 1
#         else:
#             h2h_map[teams]['draw'] += 1
            
#     df['Home_H2H_Wins'] = h2h_home_wins
#     df['Away_H2H_Wins'] = h2h_away_wins
#     df['H2H_Draws'] = h2h_draws
#     return df

# full_df = add_h2h_features(full_df)

In [16]:
def add_h2h_features(df):
    h2h_map = {} 
    
    h2h_home_wins = []
    h2h_away_wins = []
    h2h_draws = []
    
    for idx, row in df.iterrows():
        # Đảm bảo tên đội luôn được sắp xếp để tạo key duy nhất cho cặp đối đầu
        teams = tuple(sorted([row['HomeTeam'], row['AwayTeam']]))
        team_h = row['HomeTeam']
        team_a = row['AwayTeam']
        
        # Khởi tạo nếu cặp đội này lần đầu xuất hiện
        if teams not in h2h_map:
            h2h_map[teams] = {team_h: 0, team_a: 0, 'draw': 0}
        # Trường hợp đặc biệt: Đội mới thăng hạng có thể chưa có key trong dict con
        if team_h not in h2h_map[teams]: h2h_map[teams][team_h] = 0
        if team_a not in h2h_map[teams]: h2h_map[teams][team_a] = 0
        
        # 1. LẤY DỮ LIỆU LỊCH SỬ (Trước khi trận này diễn ra)
        # Đây là dữ liệu Model sẽ dùng để dự đoán
        current_h2h = h2h_map[teams]
        h2h_home_wins.append(current_h2h.get(team_h, 0))
        h2h_away_wins.append(current_h2h.get(team_a, 0))
        h2h_draws.append(current_h2h['draw'])
        
        # 2. CHỈ CẬP NHẬT KẾT QUẢ NẾU TRẬN ĐẤU ĐÃ CÓ KẾT QUẢ
        # Dùng pd.notna() để bỏ qua các trận tuần tới (FTR = NaN)
        if pd.notna(row['FTR']):
            if row['FTR'] == 'H':
                h2h_map[teams][team_h] += 1
            elif row['FTR'] == 'A':
                h2h_map[teams][team_a] += 1
            elif row['FTR'] == 'D': # Ghi rõ 'D' cho chắc chắn
                h2h_map[teams]['draw'] += 1
            
    df['Home_H2H_Wins'] = h2h_home_wins
    df['Away_H2H_Wins'] = h2h_away_wins
    df['H2H_Draws'] = h2h_draws
    return df
full_df = add_h2h_features(full_df)

In [17]:
full_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Rank_Diff,Home_Shot_Efficiency,Away_Shot_Efficiency,Home_Defense_Ratio,Away_Defense_Ratio,Home_Attack_Pressure,Away_Attack_Pressure,Home_H2H_Wins,Away_H2H_Wins,H2H_Draws
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,...,-1,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [18]:
full_df.tail(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,Rank_Diff,Home_Shot_Efficiency,Away_Shot_Efficiency,Home_Defense_Ratio,Away_Defense_Ratio,Home_Attack_Pressure,Away_Attack_Pressure,Home_H2H_Wins,Away_H2H_Wins,H2H_Draws
7932,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-6,0.382353,0.294117,0.333333,0.235293,5.440,3.12,5,3,1
7933,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-4,0.333333,0.340425,0.150000,0.357142,4.220,3.80,3,1,1
7934,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-8,0.282608,0.347826,0.266666,0.136363,3.325,3.62,7,1,1
7935,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,8,0.370370,0.439394,0.304347,0.150000,3.880,5.26,2,7,8
7936,2026-05-02 14:00:00,Wolves,Sunderland,NaN,NaN,None,NaN,NaN,NaN,NaN,...,9,0.297872,0.368421,0.357142,0.230769,2.860,4.20,4,2,1
7937,2026-05-02 16:30:00,Arsenal,Fulham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-10,0.294117,0.214286,0.294117,0.210526,4.660,3.84,18,4,7
7938,2026-05-03 13:00:00,Bournemouth,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-6,0.280000,0.265306,0.374999,0.160000,4.740,2.90,4,6,7
7939,2026-05-03 14:30:00,Man United,Liverpool,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-2,0.388889,0.337500,0.312499,0.240000,5.625,5.82,17,13,11
7940,2026-05-03 18:00:00,Aston Villa,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-14,0.367647,0.428571,0.423076,0.499999,4.820,5.94,9,19,7
7941,2026-05-04 14:00:00,Chelsea,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,...,-8,0.178082,0.396226,0.343749,0.222222,3.800,4.14,3,1,3


# ELO

In [19]:
import pandas as pd
import numpy as np

def calculate_elo(df, k_factor=30, home_advantage=100):
    # 1. Khởi tạo dictionary lưu điểm ELO hiện tại của các đội
    # Mặc định mỗi đội là 1500
    all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()
    elo_ratings = {team: 1500 for team in all_teams}
    
    # Danh sách để lưu ELO TRƯỚC trận đấu (để đưa vào Feature)
    home_elo_before = []
    away_elo_before = []
    
    # Duyệt qua từng trận đấu theo thứ tự thời gian
    for idx, row in df.iterrows():
        home_team = row['HomeTeam']
        away_team = row['AwayTeam']
        result = row['FTR']
        
        # Lấy điểm ELO hiện tại
        r_h = elo_ratings[home_team]
        r_a = elo_ratings[away_team]
        
        # Lưu lại điểm TRƯỚC trận để làm Feature dự đoán
        home_elo_before.append(r_h)
        away_elo_before.append(r_a)
        
        # 2. Tính toán xác suất mong đợi (Expected Score)
        # Có tính đến lợi thế sân nhà (home_advantage)
        e_h = 1 / (1 + 10 ** ((r_a - (r_h + home_advantage)) / 400))
        e_a = 1 - e_h
        
        # 3. Xác định kết quả thực tế (Actual Score)
        if result == 'H':
            s_h, s_a = 1, 0
        elif result == 'A':
            s_h, s_a = 0, 1
        else: # Hòa
            s_h, s_a = 0.5, 0.5
            
        # 4. Cập nhật điểm ELO sau trận đấu
        elo_ratings[home_team] = r_h + k_factor * (s_h - e_h)
        elo_ratings[away_team] = r_a + k_factor * (s_a - e_a)
        
    # Thêm vào dataframe chính
    df['Home_Elo'] = home_elo_before
    df['Away_Elo'] = away_elo_before
    df['Elo_Diff'] = df['Home_Elo'] - df['Away_Elo']
    
    return df, elo_ratings

# Chạy hàm
full_df, final_elo = calculate_elo(full_df)

# Kiểm tra thử Top 5 đội có ELO cao nhất hiện tại (cuối dữ liệu)
print("Top đội mạnh nhất dựa trên ELO:")
sorted_elo = sorted(final_elo.items(), key=lambda x: x[1], reverse=True)
print(sorted_elo[:5])

Top đội mạnh nhất dựa trên ELO:
[('Man City', 1818.8888470725315), ('Arsenal', 1789.2241112206189), ('Liverpool', 1716.128414568553), ('Aston Villa', 1682.904662574648), ('Man United', 1674.5278909609212)]


# AGGRESSION

In [20]:
# def calculate_advanced_aggression(df):
#     data = df.copy()
#     # 1. Tính điểm Aggression thô cho từng đội trong mỗi trận
#     data['Home_Points'] = (data['HF'] * 0.5) + data['HY'] + (data['HR'] * 3)
#     data['Away_Points'] = (data['AF'] * 0.5) + data['AY'] + (data['AR'] * 3)
#     data['Total_Match_Points'] = data['Home_Points'] + data['Away_Points']

#     def get_features(row):
#         home, away, date = row['HomeTeam'], row['AwayTeam'], row['Date']
        
#         # --- Phần 1: H2H Rivalry (Duyên nợ) ---
#         h2h = data[
#             ((data['HomeTeam'] == home) & (data['AwayTeam'] == away) |
#              (data['HomeTeam'] == away) & (data['AwayTeam'] == home)) &
#             (data['Date'] < date)
#         ].sort_values('Date', ascending=False).head(5)
        
#         h2h_idx = h2h['Total_Match_Points'].mean() if not h2h.empty else data['Total_Match_Points'].mean()

#         # --- Phần 2: Team Style (Phong cách lăn xả của từng đội) ---
#         # Lấy 5 trận gần nhất của Đội Nhà (bất kể sân nhà hay khách)
#         home_recent = data[
#             ((data['HomeTeam'] == home) | (data['AwayTeam'] == home)) & (data['Date'] < date)
#         ].sort_values('Date', ascending=False).head(5)
        
#         # Tính điểm trung bình đội nhà nhận được trong 5 trận đó
#         def calc_team_pts(df_recent, team):
#             pts = []
#             for _, r in df_recent.iterrows():
#                 pts.append(r['Home_Points'] if r['HomeTeam'] == team else r['Away_Points'])
#             return np.mean(pts) if pts else data['Home_Points'].mean()

#         home_style = calc_team_pts(home_recent, home)
        
#         # Làm tương tự cho Đội Khách
#         away_recent = data[
#             ((data['HomeTeam'] == away) | (data['AwayTeam'] == away)) & (data['Date'] < date)
#         ].sort_values('Date', ascending=False).head(5)
#         away_style = calc_team_pts(away_recent, away)

#         return pd.Series([h2h_idx, home_style, away_style])

#     data[['H2H_Rivalry', 'Home_Style_Agg', 'Away_Style_Agg']] = data.apply(get_features, axis=1)
#     return data

# full_df = calculate_advanced_aggression(full_df)

In [21]:
def calculate_advanced_aggression(df):
    data = df.copy()
    
    # 1. Tính điểm Aggression (Chỉ những trận đã có dữ liệu thẻ phạt)
    # Dùng fillna(0) để các trận tương lai không làm hỏng phép tính mean() sau này
    data['Home_Points'] = (data['HF'].fillna(0) * 0.5) + data['HY'].fillna(0) + (data['HR'].fillna(0) * 3)
    data['Away_Points'] = (data['AF'].fillna(0) * 0.5) + data['AY'].fillna(0) + (data['AR'].fillna(0) * 3)
    data['Total_Match_Points'] = data['Home_Points'] + data['Away_Points']

    # Tính giá trị trung bình toàn giải (chỉ tính trên các trận đã đá)
    # Lấy những trận mà FTR không phải None/NaN
    past_matches = data[data['FTR'].notna()]
    global_mean_total = past_matches['Total_Match_Points'].mean()
    global_mean_home = past_matches['Home_Points'].mean()
    global_mean_away = past_matches['Away_Points'].mean()

    def get_features(row):
        home, away, date = row['HomeTeam'], row['AwayTeam'], row['Date']
        
        # --- Phần 1: H2H Rivalry ---
        h2h = data[
            ((data['HomeTeam'] == home) & (data['AwayTeam'] == away) |
             (data['HomeTeam'] == away) & (data['AwayTeam'] == home)) &
            (data['Date'] < date)
        ].sort_values('Date', ascending=False).head(5)
        
        # Nếu không có H2H, dùng trung bình toàn giải của các trận ĐÃ ĐÁ
        h2h_idx = h2h['Total_Match_Points'].mean() if not h2h.empty else global_mean_total

        # --- Phần 2: Team Style ---
        def calc_team_pts(team, current_date, default_val):
            recent = data[
                ((data['HomeTeam'] == team) | (data['AwayTeam'] == team)) & 
                (data['Date'] < current_date) &
                (data['FTR'].notna()) # QUAN TRỌNG: Chỉ lấy phong độ từ các trận đã kết thúc
            ].sort_values('Date', ascending=False).head(5)
            
            pts = []
            for _, r in recent.iterrows():
                pts.append(r['Home_Points'] if r['HomeTeam'] == team else r['Away_Points'])
            return np.mean(pts) if pts else default_val

        home_style = calc_team_pts(home, date, global_mean_home)
        away_style = calc_team_pts(away, date, global_mean_away)

        return pd.Series([h2h_idx, home_style, away_style])

    # Thực hiện apply
    data[['H2H_Rivalry', 'Home_Style_Agg', 'Away_Style_Agg']] = data.apply(get_features, axis=1)
    return data

In [22]:
full_df = calculate_advanced_aggression(full_df)

In [23]:
full_df.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,H2H_Draws,Home_Elo,Away_Elo,Elo_Diff,Home_Points,Away_Points,Total_Match_Points,H2H_Rivalry,Home_Style_Agg,Away_Style_Agg
0,2005-08-13,Aston Villa,Bolton,2.0,2.0,D,3.0,13.0,0.0,2.0,...,0,1500.0,1500.0,0.0,7.0,10.0,17.0,14.89328,7.162443,7.730837
1,2005-08-13,Everton,Man United,0.0,2.0,A,10.0,12.0,3.0,1.0,...,0,1500.0,1500.0,0.0,10.5,8.0,18.5,14.89328,7.162443,7.730837
2,2005-08-13,Fulham,Birmingham,0.0,0.0,D,15.0,7.0,1.0,2.0,...,0,1500.0,1500.0,0.0,7.0,8.5,15.5,14.89328,7.162443,7.730837
3,2005-08-13,Man City,West Brom,0.0,0.0,D,15.0,13.0,2.0,3.0,...,0,1500.0,1500.0,0.0,8.5,8.5,17.0,14.89328,7.162443,7.730837
4,2005-08-13,Middlesbrough,Liverpool,0.0,0.0,D,4.0,16.0,2.0,3.0,...,0,1500.0,1500.0,0.0,13.5,8.5,22.0,14.89328,7.162443,7.730837


In [24]:
full_df.tail(10)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HY,AY,...,H2H_Draws,Home_Elo,Away_Elo,Elo_Diff,Home_Points,Away_Points,Total_Match_Points,H2H_Rivalry,Home_Style_Agg,Away_Style_Agg
7932,2026-04-27 19:00:00,Man United,Brentford,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1683.471410,1630.242527,53.228883,0.0,0.0,0.0,14.7,10.0,5.1
7933,2026-05-01 19:00:00,Leeds,Burnley,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1577.177298,1398.061675,179.115622,0.0,0.0,0.0,13.5,7.1,6.9
7934,2026-05-02 14:00:00,Brentford,West Ham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1636.459972,1552.302039,84.157933,0.0,0.0,0.0,13.0,5.1,7.6
7935,2026-05-02 14:00:00,Newcastle,Brighton,NaN,NaN,None,NaN,NaN,NaN,NaN,...,8,1588.750456,1666.497687,-77.747231,0.0,0.0,0.0,16.5,8.2,8.3
7936,2026-05-02 14:00:00,Wolves,Sunderland,NaN,NaN,None,NaN,NaN,NaN,NaN,...,1,1470.307459,1554.981935,-84.674476,0.0,0.0,0.0,12.4,8.1,10.2
7937,2026-05-02 16:30:00,Arsenal,Fulham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,7,1799.716810,1598.717108,200.999702,0.0,0.0,0.0,11.8,6.8,6.3
7938,2026-05-03 13:00:00,Bournemouth,Crystal Palace,NaN,NaN,None,NaN,NaN,NaN,NaN,...,7,1661.227319,1599.703765,61.523553,0.0,0.0,0.0,18.2,7.5,9.0
7939,2026-05-03 14:30:00,Man United,Liverpool,NaN,NaN,None,NaN,NaN,NaN,NaN,...,11,1677.253965,1713.402341,-36.148375,0.0,0.0,0.0,15.7,10.0,6.6
7940,2026-05-03 18:00:00,Aston Villa,Tottenham,NaN,NaN,None,NaN,NaN,NaN,NaN,...,7,1693.878390,1470.025945,223.852445,0.0,0.0,0.0,14.5,5.5,8.9
7941,2026-05-04 14:00:00,Chelsea,Nott'm Forest,NaN,NaN,None,NaN,NaN,NaN,NaN,...,3,1627.550979,1574.161672,53.389306,0.0,0.0,0.0,18.2,6.2,7.0


In [25]:
full_df.to_csv('/kaggle/working/epl_full_ft.csv')

### FEATURE COLUMNS

In [26]:
FEATURE_COLUMNS = [
    'Month_Sin', 'Month_Cos',
    'Matchweek',
    'Home_Rank', 'Away_Rank', 'Rank_Diff',
    'Home_Shot_Efficiency', 'Away_Shot_Efficiency',
    'Home_Defense_Ratio', 'Away_Defense_Ratio',
    'Home_Attack_Pressure', 'Away_Attack_Pressure',
    'Home_H2H_Wins', 'Away_H2H_Wins', 'H2H_Draws',
    'Home_Elo', 'Away_Elo', 'Elo_Diff',
    'H2H_Rivalry', 'Home_Style_Agg', 'Away_Style_Agg'
]

TARGET_COLUMNS = ['FTHG','FTAG','FTR','HS','AS','HY','AY','HST','AST','HC','AC','HR', 'AR','HF','AF']
TARGET_COLUMN = 'FTR'

# Train/Test SPLIT

In [27]:

# Filter to rows with complete features
df_model = full_df.dropna(subset=FEATURE_COLUMNS).copy()
print(f"Matches with complete features: {len(df_model)} / {len(full_df)}")

for col in FEATURE_COLUMNS:
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

df_model[FEATURE_COLUMNS] = df_model[FEATURE_COLUMNS].fillna(0)

# Time-based split
train_df = df_model[df_model['Season'] != TEST_SEASON].copy()
test_df = df_model[df_model['Season'] == TEST_SEASON].copy()

print(f"\nTraining set: {len(train_df)} matches")
print(f"  Seasons: {train_df['Season'].unique().tolist()}")
print(f"\nTest set: {len(test_df)} matches")
print(f"  Season: {TEST_SEASON}")

# Prepare X and y
X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET_COLUMN]

X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

print(f"\nFeature matrix shape: {X_train.shape}")

Matches with complete features: 7908 / 7942

Training set: 7566 matches
  Seasons: ['2005-2006', '2006-2007', '2007-2008', '2008-2009', '2009-2010', '2010-2011', '2011-2012', '2012-2013', '2013-2014', '2014-2015', '2015-2016', '2016-2017', '2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']

Test set: 342 matches
  Season: 2025-2026

Feature matrix shape: (7566, 21)


In [28]:
# =============================================================================
# CELL 8: Create Validation Split for Early Stopping
# =============================================================================

# Use last portion of training data as validation (time-based)
val_size = int(len(X_train) * VALIDATION_SPLIT)
X_train_fit = X_train.iloc[:-val_size]
y_train_fit = y_train.iloc[:-val_size]
X_val = X_train.iloc[-val_size:]
y_val = y_train.iloc[-val_size:]

print(f"Training (fit): {len(X_train_fit)} matches")
print(f"Validation: {len(X_val)} matches")
print(f"Test: {len(X_test)} matches")

Training (fit): 6053 matches
Validation: 1513 matches
Test: 342 matches


In [29]:
from sklearn.preprocessing import LabelEncoder

In [30]:
le = LabelEncoder()
y_train_fit = le.fit_transform(y_train_fit.values.ravel())
y_val = le.transform(y_val.values.ravel())

In [31]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Tính toán trọng số cho 3 lớp (0, 1, 2)
classes = np.unique(y_train_fit)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_fit)
class_weights_dict = dict(zip(classes, weights))

print(f"Trọng số các lớp: {class_weights_dict}")
sample_weights = np.array([class_weights_dict[cls] for cls in y_train_fit])

Trọng số các lớp: {np.int64(0): np.float64(1.129712579320642), np.int64(1): np.float64(1.3642100518368268), np.int64(2): np.float64(0.7236967957914874)}


In [32]:
print("🚀 Training XGBoost Classifier...")
print(f"Parameters: {XGBOOST_PARAMS}")

# 1. Đảm bảo dữ liệu truyền vào chỉ gồm các Features đã chọn và là kiểu số
X_train_final = X_train_fit[FEATURE_COLUMNS].astype('float32')
X_val_final = X_val[FEATURE_COLUMNS].astype('float32')

# 2. Khởi tạo model
model = xgb.XGBClassifier(
    **XGBOOST_PARAMS,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS
)

# 3. Train với eval set
model.fit(
    X_train_final, y_train_fit,
    eval_set=[(X_val_final, y_val)],
    verbose=50,
    #sample_weight=sample_weights# Hiển thị tiến trình mỗi 50 vòng
)

print(f"\n✅ Training complete!")
print(f"🥇 Best iteration: {model.best_iteration}")

# 4. Hiển thị đúng tên chỉ số đo lường
if XGBOOST_PARAMS.get('objective') == 'multi:softprob':
    print(f"📉 Best validation mlogloss: {model.best_score:.4f}")
else:
    print(f"📉 Best validation score: {model.best_score:.4f}")

🚀 Training XGBoost Classifier...
Parameters: {'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': ['mlogloss', 'merror'], 'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_weight': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1}
[0]	validation_0-mlogloss:1.07124	validation_0-merror:0.55585
[50]	validation_0-mlogloss:0.97624	validation_0-merror:0.44613
[100]	validation_0-mlogloss:0.97765	validation_0-merror:0.45539
[103]	validation_0-mlogloss:0.97804	validation_0-merror:0.45340

✅ Training complete!
🥇 Best iteration: 53
📉 Best validation mlogloss: 0.4435


In [33]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Dự đoán trên tập Validation
y_pred = model.predict(X_val_final)

# 2. Tính Accuracy
acc = accuracy_score(y_val, y_pred)

print(f"📊 Accuracy trên tập Validation: {acc*100:.2f}%")
print("-" * 30)

# 3. Xem chi tiết đoán đúng/sai từng cửa (Thắng, Hòa, Thua)
# le.classes_ giúp dịch ngược 0, 1, 2 về lại Away, Draw, Home
print(classification_report(y_val, y_pred, target_names=le.classes_))

📊 Accuracy trên tập Validation: 55.65%
------------------------------
              precision    recall  f1-score   support

           A       0.55      0.57      0.56       491
           D       0.00      0.00      0.00       350
           H       0.56      0.84      0.67       672

    accuracy                           0.56      1513
   macro avg       0.37      0.47      0.41      1513
weighted avg       0.43      0.56      0.48      1513



In [34]:
probs_np = model.predict_proba(X_test) * 100

# 3. Chạy model để lấy nhãn dự đoán (0, 1, 2)
preds_idx = model.predict(X_test)
preds_labels = le.inverse_transform(preds_idx)

# 4. Tạo DataFrame xác suất
cols = [f'{c} %' for c in le.classes_]
df_probs = pd.DataFrame(probs_np, columns=cols)
df_probs['Predict'] = preds_labels
teams_info = test_df[['Date', 'HomeTeam', 'AwayTeam', 'Elo_Diff']].reset_index(drop=True)
final_display = pd.concat([teams_info, df_probs.round(2)], axis=1)

print(f"✅ Đã xử lý xong {len(final_display)} trận cuối mùa {TEST_SEASON}.")

display(final_display.tail(10).style.background_gradient(subset=cols, cmap='YlGn'))

✅ Đã xử lý xong 342 trận cuối mùa 2025-2026.


,Date,HomeTeam,AwayTeam,Elo_Diff,A %,D %,H %,Predict
332,2026-04-27 19:00:00,Man United,Brentford,53.228883,22.090000,26.840000,51.070000,H
333,2026-05-01 19:00:00,Leeds,Burnley,179.115622,19.450001,20.719999,59.830002,H
334,2026-05-02 14:00:00,Brentford,West Ham,84.157933,26.040001,25.110001,48.849998,H
335,2026-05-02 14:00:00,Newcastle,Brighton,-77.747231,42.970001,25.549999,31.490000,A
336,2026-05-02 14:00:00,Wolves,Sunderland,-84.674476,39.580002,26.110001,34.310001,A
337,2026-05-02 16:30:00,Arsenal,Fulham,200.999702,15.840000,14.360000,69.800003,H
338,2026-05-03 13:00:00,Bournemouth,Crystal Palace,61.523553,25.330000,27.320000,47.349998,H
339,2026-05-03 14:30:00,Man United,Liverpool,-36.148375,27.100000,31.360001,41.540001,H
340,2026-05-03 18:00:00,Aston Villa,Tottenham,223.852445,17.320000,17.680000,65.000000,H
341,2026-05-04 14:00:00,Chelsea,Nott'm Forest,53.389306,25.940001,25.370001,48.689999,H


In [35]:
final_display.to_csv('/kaggle/working/final.csv')

# LightGBM

In [36]:
import lightgbm as lgb

# 1. Cấu hình Parameters cho LightGBM
LGBM_PARAMS = {
    'objective': 'multiclass',        # Tương đương multi:softprob
    'num_class': 3,                   # 0 (Away), 1 (Draw), 2 (Home)
    'metric': ['multi_logloss', 'multi_error'], # Thước đo mlogloss và merror
    'max_depth': 4,                   
    'learning_rate': 0.05,             
    'n_estimators': 500,               
    'min_child_samples': 10,          # Tương đương min_child_weight (số record tối thiểu trong 1 lá)
    'subsample': 0.8,                 
    'colsample_bytree': 0.8,           
    'reg_alpha': 0.1,                 # L1 regularization
    'reg_lambda': 1.0,                # L2 regularization
    'random_state': 42,
    'n_jobs': -1,
    'importance_type': 'gain',        # Tính quan trọng dựa trên Gain (giống XGBoost)
    'verbosity': -1                   # Tắt các cảnh báo không cần thiết
}

# ----- Early Stopping -----
EARLY_STOPPING_ROUNDS = 50
print("🚀 Training LightGBM Classifier...")
print(f"Parameters: {LGBM_PARAMS}")

# 2. Khởi tạo model
# Lưu ý: LightGBM dùng 'early_stopping_round' trong hàm fit hoặc thông qua callback
model = lgb.LGBMClassifier(**LGBM_PARAMS)

# 3. Train với eval set
# Trong LightGBM hiện đại, early_stopping được truyền trực tiếp vào fit() 
# hoặc qua callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS)]
model.fit(
    X_train_final, y_train_fit,
    eval_set=[(X_val_final, y_val)],
    eval_metric=['multi_logloss', 'multi_error'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=50) # Tương đương verbose=50
    ]
)

print(f"\n✅ Training complete!")
print(f"🥇 Best iteration: {model.best_iteration_}")

# 4. Hiển thị kết quả
# Lưu ý: model.best_score_ của LGBM trả về dictionary các bộ metrics
best_score = model.best_score_['valid_0']['multi_logloss']
print(f"📉 Best validation mlogloss: {best_score:.4f}")

🚀 Training LightGBM Classifier...
Parameters: {'objective': 'multiclass', 'num_class': 3, 'metric': ['multi_logloss', 'multi_error'], 'max_depth': 4, 'learning_rate': 0.05, 'n_estimators': 500, 'min_child_samples': 10, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_alpha': 0.1, 'reg_lambda': 1.0, 'random_state': 42, 'n_jobs': -1, 'importance_type': 'gain', 'verbosity': -1}
Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 0.978664	valid_0's multi_error: 0.456048
Early stopping, best iteration is:
[44]	valid_0's multi_logloss: 0.978163	valid_0's multi_error: 0.455387

✅ Training complete!
🥇 Best iteration: 44
📉 Best validation mlogloss: 0.9782


In [37]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Dự đoán trên tập Validation
y_pred = model.predict(X_val_final)

# 2. Tính Accuracy
acc = accuracy_score(y_val, y_pred)

print(f"📊 Accuracy trên tập Validation: {acc*100:.2f}%")
print("-" * 30)

# 3. Xem chi tiết đoán đúng/sai từng cửa (Thắng, Hòa, Thua)
# le.classes_ giúp dịch ngược 0, 1, 2 về lại Away, Draw, Home
print(classification_report(y_val, y_pred, target_names=le.classes_))

📊 Accuracy trên tập Validation: 54.46%
------------------------------
              precision    recall  f1-score   support

           A       0.54      0.54      0.54       491
           D       0.33      0.00      0.01       350
           H       0.55      0.83      0.66       672

    accuracy                           0.54      1513
   macro avg       0.47      0.46      0.40      1513
weighted avg       0.50      0.54      0.47      1513



# TabPFN

In [38]:
!pip install tabpfn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.3/228.3 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-ml-py
    Found existing installation: nvidia-ml-py 13.590.44
    Uninstalling nvidia-ml-py-13.590.44:
      Successfully uninstalled nvidia-ml-py-13.590.44
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, whic

In [ ]:
import os
# os.environ["HF_TOKEN"] = 
# os.environ["TABPFN_TOKEN"] =
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 TabPFN sẽ chạy trên: {device}")

🚀 TabPFN sẽ chạy trên: cuda


In [40]:
!pip install tabpfn-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.0 MB/s eta 0:00:00


In [ ]:
#from tabpfn import TabPFNClassifier
from tabpfn_client import TabPFNClassifier, set_access_token, TabPFNRegressor, TabPFNClassifier

# set_access_token("")

# 1. Cấu hình Parameters cho TabPFN
# Lưu ý: TabPFN hầu như không cần tinh chỉnh (tuning) như XGBoost

print("🚀 Initializing TabPFN Classifier...")
print("Lưu ý: TabPFN không cần quá trình huấn luyện (training) truyền thống.")

# 2. Khởi tạo model
# TabPFN được thiết kế để hoạt động tốt nhất "out-of-the-box"
model = TabPFNClassifier()

# 3. "Huấn luyện" model
# Với TabPFN, hàm fit thực chất là nạp dữ liệu vào bộ nhớ của Transformer
# Nó không hỗ trợ Early Stopping vì nó không chạy qua các iteration (vòng lặp)
model.fit(X_train_final, y_train_fit)

print(f"✅ Model is ready for prediction!")

# 4. Dự đoán xác suất (cho Monte Carlo)
# Giống như XGBoost, bạn dùng predict_proba để lấy xác suất 3 lớp
probs = model.predict_proba(X_val_final)

# TabPFN không có model.best_score_ như XGB hay LGBM 
# vì nó không phải là mô hình lặp (iterative)
from sklearn.metrics import log_loss
val_logloss = log_loss(y_val, probs)
print(f"📉 Validation mlogloss: {val_logloss:.4f}")

🚀 Initializing TabPFN Classifier...
Lưu ý: TabPFN không cần quá trình huấn luyện (training) truyền thống.
✅ Model is ready for prediction!


Processing: 100%|██████████| [00:01<00:00]

📉 Validation mlogloss: 0.9647


In [42]:
from sklearn.metrics import accuracy_score, classification_report

# 1. Dự đoán trên tập Validation
y_pred = model.predict(X_val_final)

# 2. Tính Accuracy
acc = accuracy_score(y_val, y_pred)

print(f"📊 Accuracy trên tập Validation: {acc*100:.2f}%")
print("-" * 30)

# 3. Xem chi tiết đoán đúng/sai từng cửa (Thắng, Hòa, Thua)
# le.classes_ giúp dịch ngược 0, 1, 2 về lại Away, Draw, Home
print(classification_report(y_val, y_pred, target_names=le.classes_))

Processing: 100%|██████████| [00:01<00:00]

📊 Accuracy trên tập Validation: 55.78%
------------------------------
              precision    recall  f1-score   support

           A       0.54      0.60      0.57       491
           D       0.00      0.00      0.00       350
           H       0.57      0.82      0.67       672

    accuracy                           0.56      1513
   macro avg       0.37      0.47      0.41      1513
weighted avg       0.43      0.56      0.48      1513

